In [1]:
import pandas as pd
import numpy as np
import os

train = pd.read_parquet(
    "data/artifacts/03_train.parquet"
)

validation = pd.read_parquet(
    "data/artifacts/03_validation.parquet"
)

test = pd.read_parquet(
    "data/artifacts/03_test.parquet"
)

print("Train:", train.shape)
print("Validation:", validation.shape)
print("Test:", test.shape)

Train: (67533, 42)
Validation: (14471, 42)
Test: (14472, 42)


In [2]:
def create_time_features(df):

    df = df.copy()

    df["order_purchase_timestamp"] = pd.to_datetime(
        df["order_purchase_timestamp"]
    )

    df["purchase_month"] = (
        df["order_purchase_timestamp"].dt.month
    )

    df["purchase_weekday"] = (
        df["order_purchase_timestamp"].dt.dayofweek
    )

    df["purchase_hour"] = (
        df["order_purchase_timestamp"].dt.hour
    )

    # الوقت المتوقع للتوصيل من لحظة الشراء
    df["estimated_delivery_days"] = (
        pd.to_datetime(df["order_estimated_delivery_date"])
        - df["order_purchase_timestamp"]
    ).dt.total_seconds() / (24 * 3600)

    return df

In [4]:
train = create_time_features(train)
validation = create_time_features(validation)
test = create_time_features(test)

In [5]:
feature_columns = [
    "seller_customer_distance_km",
    "total_freight",
    "purchase_month",
    "purchase_weekday",
    "purchase_hour",
    "estimated_delivery_days",
    "customer_state",
    "total_price",
    "num_items",
    "num_unique_products",
    "num_unique_sellers",
    "total_product_weight_g",
    "total_product_volume_cm3"
]

In [6]:
target = "is_delayed"

In [7]:
X_train = train[feature_columns].copy()
y_train = train[target].copy()

X_validation = validation[feature_columns].copy()
y_validation = validation[target].copy()

X_test = test[feature_columns].copy()
y_test = test[target].copy()

In [8]:
numeric_features = [
    "seller_customer_distance_km",
    "total_freight",
    "purchase_month",
    "purchase_weekday",
    "purchase_hour",
    "estimated_delivery_days",
    "total_price",
    "num_items",
    "num_unique_products",
    "num_unique_sellers",
    "total_product_weight_g",
    "total_product_volume_cm3"
]

categorical_features = [
    "customer_state"
]

In [9]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

In [10]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

In [11]:
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

In [14]:
preprocessor = ColumnTransformer([
    (
        "numeric",
        numeric_pipeline,
        numeric_features
    ),
    (
        "categorical",
        categorical_pipeline,
        categorical_features
    )
])

In [15]:
X_train_processed = preprocessor.fit_transform(
    X_train
)

X_validation_processed = preprocessor.transform(
    X_validation
)

X_test_processed = preprocessor.transform(
    X_test
)

In [16]:
feature_names = preprocessor.get_feature_names_out()

print("عدد Features بعد preprocessing:",
      len(feature_names))

عدد Features بعد preprocessing: 39


In [17]:
X_train_processed = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index
)

X_validation_processed = pd.DataFrame(
    X_validation_processed,
    columns=feature_names,
    index=X_validation.index
)

X_test_processed = pd.DataFrame(
    X_test_processed,
    columns=feature_names,
    index=X_test.index
)

In [18]:
print("Processed Train:",
      X_train_processed.shape)

print("Processed Validation:",
      X_validation_processed.shape)

print("Processed Test:",
      X_test_processed.shape)

Processed Train: (67533, 39)
Processed Validation: (14471, 39)
Processed Test: (14472, 39)


In [19]:
print(
    "Missing values in processed Train:",
    X_train_processed.isna().sum().sum()
)

print(
    "Missing values in processed Validation:",
    X_validation_processed.isna().sum().sum()
)

print(
    "Missing values in processed Test:",
    X_test_processed.isna().sum().sum()
)

Missing values in processed Train: 0
Missing values in processed Validation: 0
Missing values in processed Test: 0


In [20]:
os.makedirs(
    "data/artifacts/features",
    exist_ok=True
)

In [21]:
X_train_processed.to_parquet(
    "data/artifacts/features/05_X_train.parquet",
    index=False
)

X_validation_processed.to_parquet(
    "data/artifacts/features/05_X_validation.parquet",
    index=False
)

X_test_processed.to_parquet(
    "data/artifacts/features/05_X_test.parquet",
    index=False
)

In [22]:
y_train.to_frame().to_parquet(
    "data/artifacts/features/05_y_train.parquet",
    index=False
)

y_validation.to_frame().to_parquet(
    "data/artifacts/features/05_y_validation.parquet",
    index=False
)

y_test.to_frame().to_parquet(
    "data/artifacts/features/05_y_test.parquet",
    index=False
)

In [23]:
import joblib

joblib.dump(
    preprocessor,
    "data/artifacts/features/05_preprocessor.joblib"
)

['data/artifacts/features/05_preprocessor.joblib']

In [24]:
print("Train:", X_train_processed.shape)
print("Validation:", X_validation_processed.shape)
print("Test:", X_test_processed.shape)

print(
    "Missing Train:",
    X_train_processed.isna().sum().sum()
)

print(
    "Missing Validation:",
    X_validation_processed.isna().sum().sum()
)

print(
    "Missing Test:",
    X_test_processed.isna().sum().sum()
)

Train: (67533, 39)
Validation: (14471, 39)
Test: (14472, 39)
Missing Train: 0
Missing Validation: 0
Missing Test: 0


In [25]:
print(feature_names)

['numeric__seller_customer_distance_km' 'numeric__total_freight'
 'numeric__purchase_month' 'numeric__purchase_weekday'
 'numeric__purchase_hour' 'numeric__estimated_delivery_days'
 'numeric__total_price' 'numeric__num_items'
 'numeric__num_unique_products' 'numeric__num_unique_sellers'
 'numeric__total_product_weight_g' 'numeric__total_product_volume_cm3'
 'categorical__customer_state_AC' 'categorical__customer_state_AL'
 'categorical__customer_state_AM' 'categorical__customer_state_AP'
 'categorical__customer_state_BA' 'categorical__customer_state_CE'
 'categorical__customer_state_DF' 'categorical__customer_state_ES'
 'categorical__customer_state_GO' 'categorical__customer_state_MA'
 'categorical__customer_state_MG' 'categorical__customer_state_MS'
 'categorical__customer_state_MT' 'categorical__customer_state_PA'
 'categorical__customer_state_PB' 'categorical__customer_state_PE'
 'categorical__customer_state_PI' 'categorical__customer_state_PR'
 'categorical__customer_state_RJ' 'cat

In [26]:
print("عدد Features:", len(feature_names))

عدد Features: 39


In [27]:
feature_list = pd.DataFrame({
    "feature_name": feature_names
})

feature_list.to_csv(
    "data/artifacts/features/05_feature_list.csv",
    index=False
)

print("Feature list saved.")

Feature list saved.
